#1.extracting the data from the website

In [1]:
import requests
from bs4 import BeautifulSoup

cateogry_urls = {"Travel":"https://books.toscrape.com/catalogue/category/books/travel_2/index.html",
"Mystery":"https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
"Historical Fiction":"https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
"Sequential Art":"https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html"}

all_books = []

for cateogry_name, url in cateogry_urls.items():
  response = requests.get(url)
  soup = BeautifulSoup(response.content, "html.parser")

  books = soup.find_all('article', class_='product_pod')

  print(f"Found {len(books)} books on this page")


  for book in books:
    # Title is inside the <h3><a title="..."> tag
    title = book.h3.a['title']

    # Price is inside <p class="price_color">
    price_text = book.find('p', class_='price_color').text

    # Star rating is stored as a CSS class, e.g. class="star-rating Three"
    rating_class = book.find('p', class_='star-rating')['class']
    # rating_class will look like ['star-rating', 'Three']

    # Availability text is inside <p class="instock availability">
    availability_text = book.find('p', class_='instock availability').text.strip()

    all_books.append({
        "title":title,
        "price_text":price_text,
        "rating_class":rating_class,
        "availability_text":availability_text,
        "category": cateogry_name

    })
print(f"\n total books collected:{len(all_books)}")

for book in all_books:
  print(f"\n title: {book['title']} || price_text:{book['price_text']} || rating_class:{book['rating_class']} || category: {book['category']}")

Found 11 books on this page
Found 20 books on this page
Found 20 books on this page
Found 20 books on this page

 total books collected:71

 title: It's Only the Himalayas || price_text:£45.17 || rating_class:['star-rating', 'Two'] || category: Travel

 title: Full Moon over Noah’s Ark: An Odyssey to Mount Ararat and Beyond || price_text:£49.43 || rating_class:['star-rating', 'Four'] || category: Travel

 title: See America: A Celebration of Our National Parks & Treasured Sites || price_text:£48.87 || rating_class:['star-rating', 'Three'] || category: Travel

 title: Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel || price_text:£36.94 || rating_class:['star-rating', 'Two'] || category: Travel

 title: Under the Tuscan Sun || price_text:£37.33 || rating_class:['star-rating', 'Three'] || category: Travel

 title: A Summer In Europe || price_text:£44.34 || rating_class:['star-rating', 'Two'] || category: Travel

 title: The Great Railway Bazaar || price_text:£30.54 || 

#2.Cleaning the scraped fields into proper types

In [2]:
import pandas as pd

df = pd.DataFrame(all_books)

#cleaning price
df["price_gbp"] = df["price_text"].str.replace("£", "").astype(float)

#cleaning rating
rating_map = {"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
df["rating"] = df["rating_class"].apply(lambda x: rating_map.get(x[1], None))

#clean availability
df["in_stock"] = df["availability_text"].str.contains("In stock", case=False)

#checking for missing data
print("missing price_gbp",df["price_gbp"].isna().sum())
print("missing rating",df["rating"].isna().sum())

df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
df["rating"] = df["rating"].fillna(df["rating"].median())
df["rating"] = df["rating"].astype(int)

print(df[["title", "price_gbp", "rating", "in_stock", "category"]].head(10))
print(f"\n Total rows: {len(df)}")


print(df.dtypes)
print(f"missing after cleaning: {df[["price_gbp","rating"]].isna().sum().sum()}")

missing price_gbp 0
missing rating 0
                                               title  price_gbp  rating  \
0                            It's Only the Himalayas      45.17       2   
1  Full Moon over Noah’s Ark: An Odyssey to Mount...      49.43       4   
2  See America: A Celebration of Our National Par...      48.87       3   
3  Vagabonding: An Uncommon Guide to the Art of L...      36.94       2   
4                               Under the Tuscan Sun      37.33       3   
5                                 A Summer In Europe      44.34       2   
6                           The Great Railway Bazaar      30.54       1   
7                   A Year in Provence (Provence #1)      56.88       4   
8  The Road to Little Dribbling: Adventures of an...      23.21       1   
9          Neither Here nor There: Travels in Europe      38.95       3   

   in_stock category  
0      True   Travel  
1      True   Travel  
2      True   Travel  
3      True   Travel  
4      True   Travel  

#3.Convert price_gbp to a price_inr column

In [3]:
df["price_inr"] = df["price_gbp"] * 105.50

print(df[["title", "price_gbp", "price_inr", "category"]].head(10))
print(f"/n total rows: {len(df)}")
print(df[["price_inr","price_gbp"]].dtypes)

                                               title  price_gbp  price_inr  \
0                            It's Only the Himalayas      45.17   4765.435   
1  Full Moon over Noah’s Ark: An Odyssey to Mount...      49.43   5214.865   
2  See America: A Celebration of Our National Par...      48.87   5155.785   
3  Vagabonding: An Uncommon Guide to the Art of L...      36.94   3897.170   
4                               Under the Tuscan Sun      37.33   3938.315   
5                                 A Summer In Europe      44.34   4677.870   
6                           The Great Railway Bazaar      30.54   3221.970   
7                   A Year in Provence (Provence #1)      56.88   6000.840   
8  The Road to Little Dribbling: Adventures of an...      23.21   2448.655   
9          Neither Here nor There: Travels in Europe      38.95   4109.225   

  category  
0   Travel  
1   Travel  
2   Travel  
3   Travel  
4   Travel  
5   Travel  
6   Travel  
7   Travel  
8   Travel  
9   Travel 

#4.normalized SQLite schema with at least two tables

In [4]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories(
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE

)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books(
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

conn.commit()

category_map = {}
for category_name in df["category"].unique():
  cursor.execute("INSERT OR IGNORE INTO categories (category_name) VALUES (?)",(category_name,))
conn.commit()

cursor.execute("SELECT category_id,category_name FROM categories")
for cat_id, cat_name in cursor.fetchall():
  category_map[cat_name] = cat_id


for _,row in df.iterrows():
  cursor.execute("""
      INSERT INTO books(title,price_gbp,price_inr,rating,in_stock,category_id)
      VALUES(?,?,?,?,?,?)
  """,(
      row["title"],
      row["price_gbp"],
      row["price_inr"],
      row["rating"],
      int(row["in_stock"]),
      category_map[row["category"]]
  ))

conn.commit()


cursor.execute("SELECT COUNT(*) FROM books")
print("books inserted:",cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM categories")
print("categories inserted:",cursor.fetchone()[0])

books inserted: 71
categories inserted: 4


#5.insert your cleaned, converted data into this schema.

In [5]:
queries = {}
results = {}

queries["q1_select_where"] = """
SELECT title,price_gbp, rating
FROM books
WHERE price_gbp >40
"""

queries["q2_orderby_limit"] = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp ASC
LIMIT 5
"""

queries["q3_distinct"] = """
SELECT DISTINCT rating
FROM books
ORDER BY rating
"""

queries["q4_between"] = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 30
"""

queries["q5_join"] = """
SELECT books.title, books.rating, categories.category_name
FROM books
JOIN categories ON books.category_id = categories.category_id
ORDER BY books.rating DESC,books.title ASC
LIMIT 10
"""

for name, sql in queries.items():
  cursor.execute(sql)
  rows = cursor.fetchall()
  results[name] = rows
  print(f"\n--- {name} ---")
  print(sql.strip())
  print("output:")
  for row in rows:
    print(row)


--- q1_select_where ---
SELECT title,price_gbp, rating
FROM books
WHERE price_gbp >40
output:
("It's Only the Himalayas", 45.17, 2)
('Full Moon over Noah’s Ark: An Odyssey to Mount Ararat and Beyond', 49.43, 4)
('See America: A Celebration of Our National Parks & Treasured Sites', 48.87, 3)
('A Summer In Europe', 44.34, 2)
('A Year in Provence (Provence #1)', 56.88, 4)
('Sharp Objects', 47.82, 4)
('The Past Never Ends', 56.5, 4)
('The Murder of Roger Ackroyd (Hercule Poirot #4)', 44.1, 4)
('The Last Mile (Amos Decker #2)', 54.21, 2)
('A Time of Torment (Charlie Parker #14)', 48.35, 5)
('Murder at the 42nd Street Library (Raymond Ambler #1)', 54.36, 4)
('Boar Island (Anna Pigeon #19)', 59.48, 3)
("The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)", 52.3, 5)
('Tipping the Velvet', 53.74, 1)
('A Flight of Arrows (The Pathfinders #2)', 55.53, 5)
('Glory over Everything: Beyond The Kitchen House', 45.84, 3)
('The Last Painting of Sara de Vos', 55.55, 2)
('The Guernse

#6.query results into pandas DataFrames using pd.read_sql(...)

In [6]:
import pandas as pd

df_ql_sql = pd.read_sql(queries["q1_select_where"], conn)
df_q5_sql = pd.read_sql(queries["q5_join"], conn)

print("\n Query 1 (SELECT/WHERE) via pd.read_sql:")
print(df_ql_sql)

print("\nQuery 5 (JOIN) via pd.read_sql:")
print(df_q5_sql)

books_df = pd.read_sql("SELECT * FROM BOOKS", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)

merged_df = pd.merge(books_df, categories_df, on="category_id")
merged_df = merged_df.sort_values(["rating","title"], ascending =[False,True]).head(10)
merged_df =merged_df[["title", "rating", "category_name"]].reset_index(drop = True)


print("\n Query 5 (JOIN) reproduced via pd.merge:")
print(merged_df)

df_q5_sql_reset = df_q5_sql.reset_index(drop=True)

print("\nSQL JOIN result:")
print(df_q5_sql_reset)

print("\n pandas merge result:")
print(merged_df)


are_equal = df_q5_sql_reset.equals(merged_df)
print(f"\n Are the two results euqivalent? {are_equal}")


 Query 1 (SELECT/WHERE) via pd.read_sql:
                                                title  price_gbp  rating
0                             It's Only the Himalayas      45.17       2
1   Full Moon over Noah’s Ark: An Odyssey to Mount...      49.43       4
2   See America: A Celebration of Our National Par...      48.87       3
3                                  A Summer In Europe      44.34       2
4                    A Year in Provence (Provence #1)      56.88       4
5                                       Sharp Objects      47.82       4
6                                 The Past Never Ends      56.50       4
7     The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10       4
8                      The Last Mile (Amos Decker #2)      54.21       2
9              A Time of Torment (Charlie Parker #14)      48.35       5
10  Murder at the 42nd Street Library (Raymond Amb...      54.36       4
11                      Boar Island (Anna Pigeon #19)      59.48       3
12  The B